In [11]:
import pandas as pd
import numpy as np

In [17]:
ratings_csv = r"C:\Users\fkamdar\Desktop\repos\cp_nonmotor\stimuli\all_NAPs_ratings.csv"
out_csv = r"C:\Users\fkamdar\Desktop\repos\cp_nonmotor\stimuli\pilot_block_112.csv"

In [3]:
N_BINS = 8
TRIALS_PER_BIN = 14
COND_PER_BIN = 7
SEED = 42
np.random.seed(SEED)

In [19]:
# ===== LOAD DATA =====
df = pd.read_csv(ratings_csv)



In [18]:
df

,ID,Category,Nr,V/H,Description,Valence,Arousal,Width,Height,Luminance,Contrast,val_bin
0,Animals_001_h,Animals,1,h,Dead Stork,2.57,6.44,1600,1200,126.05,68.45,2-3
1,Animals_002_v,Animals,2,v,Lion,6.24,6.68,1200,1600,123.41,32.34,6-7
2,Animals_003_h,Animals,3,h,Snake,5.24,5.52,1600,1200,135.28,59.92,5-6
3,Animals_004_v,Animals,4,v,Wolf,4.50,7.02,1200,1600,122.15,75.10,4-5
4,Animals_005_h,Animals,5,h,Bat,5.31,5.82,1600,1200,131.81,59.77,5-6
...,...,...,...,...,...,...,...,...,...,...,...,...
1351,People_246_h,People,246,h,Black Eye,1.96,7.11,1600,1200,149.70,58.15,1-2
1352,People_247_v,People,247,v,Ear Puncture,4.76,6.20,1200,1600,68.08,53.97,4-5
1353,People_248_h,People,248,h,Jet Planes,6.04,5.83,1600,1200,130.01,79.28,6-7
1354,People_249_h,People,249,h,Airplane,5.64,5.67,1600,1200,129.61,84.04,5-6


In [26]:
# assign valence bins
bins = [1,2,3,4,5,6,7,8,9]
bin_labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]
df["val_bin"] = pd.cut(df["Valence"], bins=bins, labels=bin_labels, include_lowest=True)
selected = []

In [27]:
for b in bin_labels:

    pool = df[df["val_bin"] == b].copy()
    if pool["Category"].isna().any():
        n_missing = int(pool["Category"].isna().sum())
        raise ValueError(f"Bin {b} has {n_missing} missing Category values")

    if len(pool) < TRIALS_PER_BIN:
        raise ValueError(f"Bin {b} has only {len(pool)} images")

    cats = pool["Category"].unique().tolist()
    rng = np.random.default_rng(SEED + bin_labels.index(b))
    rng.shuffle(cats)

    available_counts = pool["Category"].value_counts().to_dict()
    target_counts = {cat: 0 for cat in cats}
    remaining = TRIALS_PER_BIN

    while remaining > 0:
        progressed = False
        for cat in cats:
            if target_counts[cat] < available_counts[cat]:
                target_counts[cat] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise ValueError(f"Bin {b} cannot satisfy category-balance constraint")

    sampled_parts = []
    for cat in cats:
        n_take = target_counts[cat]
        if n_take > 0:
            cat_pool = pool[pool["Category"] == cat]
            sampled_parts.append(cat_pool.sample(n_take, random_state=SEED + bin_labels.index(b)))

    sample = pd.concat(sampled_parts).sample(frac=1, random_state=SEED + bin_labels.index(b)).reset_index(drop=True)

    sample = sample.copy()
    sample["condition"] = ["FEEL"] * COND_PER_BIN + ["TONE"] * COND_PER_BIN

    selected.append(sample)

In [28]:
block = pd.concat(selected)
block = block.sample(frac=1).reset_index(drop=True)

In [29]:
block["trial"] = np.arange(1, len(block)+1)

# filename column
block["filename"] = block["ID"] + ".jpg"

block[["trial","ID","filename","Category","Valence","val_bin","condition"]].to_csv(out_csv,index=False)

print("Block generated:", out_csv)
print(block.groupby(["val_bin","condition"]).size())

Block generated: C:\Users\fkamdar\Desktop\repos\cp_nonmotor\stimuli\pilot_block_112.csv
val_bin  condition
1-2      FEEL         7
         TONE         7
2-3      FEEL         7
         TONE         7
3-4      FEEL         7
         TONE         7
4-5      FEEL         7
         TONE         7
5-6      FEEL         7
         TONE         7
6-7      FEEL         7
         TONE         7
7-8      FEEL         7
         TONE         7
8-9      FEEL         7
         TONE         7
dtype: int64


C:\Users\fkamdar\AppData\Local\Temp\ipykernel_23352\4218779652.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(block.groupby(["val_bin","condition"]).size())
